# Re-evaluate Cordoba / Greece / Canada with the current (v2.2, post-Australia) checkpoint

Cordoba, Greece, and Canada were only ever evaluated with the v2.0 checkpoint (before the
Australia training biome was added in v2.2). The Roadmap claims "+10-20 IoU ZS" from adding
Australia, but that was never actually verified on these three sites -- Chile is the only
site evaluated with the current checkpoint. This notebook re-runs all three with the v2.2
checkpoint, at both the fixed threshold (0.450) and via a per-site threshold sweep (same
question as the Chile dNBR analysis: is a low IoU a calibration problem or a discrimination
problem?), plus an AUC-ROC estimate from the swept ROC curve.

Old v2.0 numbers (for comparison, printed at the end):
- Cordoba: IoU=0.115, Precision=0.20, Recall=21%, AUC-ROC=0.738
- Greece:  IoU=0.234, Precision=0.481, Recall=31%, AUC-ROC=0.652
- Canada:  IoU=0.191, Precision=0.680, Recall=21%, AUC-ROC=0.606

**Environment**: Colab, GPU required (T4 is enough, this is inference only). Run all cells
top to bottom. ~25,817 patches total across the 3 sites -- expect roughly 45-70 minutes
depending on Drive copy speed and GPU.


In [ ]:
# [1] Install dependencies -- IDEMPOTENT, safe to re-run or re-execute via "Run all"
# after the restart below. Only installs and restarts once.
import subprocess, sys, os

needs = False
try:
    import numpy as np
    from terratorch.registry import BACKBONE_REGISTRY
    print(f'OK -- numpy={np.__version__}, terratorch ready. Continue to cell [2].')
except Exception as e:
    needs = True
    print(f'Installing ({e})...')

if needs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'terratorch', 'einops', 'timm', 'segmentation-models-pytorch'], check=True)
    print('Restarting kernel once -- after it restarts, continue from cell [2] (do NOT re-run this cell manually).')
    os.kill(os.getpid(), 9)


In [ ]:
# [2] Drive mount
from google.colab import drive
drive.mount('/content/drive')
IN_COLAB = True


In [ ]:
# [3] Imports, seed, device
import json, shutil, subprocess, random
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

from terratorch.registry import BACKBONE_REGISTRY

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}  device={DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# [4] Paths and constants
BASE    = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
CKPT    = BASE / 'models/best_prithvi_v22_burn_scar_wildfire.pth'
OUTPUTS_V22 = BASE / 'outputs' / 'v2.2'
OUTPUTS_V22.mkdir(parents=True, exist_ok=True)

BAND_IDX     = [0, 1, 2, 4, 5, 6]
PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420], dtype=np.float32)
IMG_SIZE = 224
T        = (256 - IMG_SIZE) // 2

SITES = {
    'cordoba': BASE / 'data' / 'cordoba' / 'patches',
    'greece':  BASE / 'data' / 'greece'  / 'patches',
    'canada':  BASE / 'data' / 'canada'  / 'patches',
}


In [ ]:
# [5] Architecture -- MultiScaleNeck + FPNDecoder + PrithviSegModelV2
# (copied verbatim from 15_train_v22.ipynb cell [6] -- do not change, avoids
# re-triggering the Prithvi loading errors already documented in prithvi_loading.md)
class MultiScaleNeck(nn.Module):
    def __init__(self, n_side=14, d_model=1024, fpn_ch=256):
        super().__init__()
        self.n_side = n_side
        self.lateral = nn.ModuleList([
            nn.Sequential(nn.Conv2d(d_model, fpn_ch, 1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(4)
        ])
        self.td_conv = nn.ModuleList([
            nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, padding=1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(3)
        ])
    def tok2map(self, tokens):
        B, L, D = tokens.shape
        return tokens.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)
    def forward(self, layers_out):
        feats = [self.lateral[i](self.tok2map(layers_out[idx][:, 1:, :]))
                 for i, idx in enumerate([5, 11, 17, 23])]
        out = feats[3]
        out = self.td_conv[0](feats[2] + out)
        out = self.td_conv[1](feats[1] + out)
        out = self.td_conv[2](feats[0] + out)
        return out


class FPNDecoder(nn.Module):
    def __init__(self, in_ch=256):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviSegModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BACKBONE_REGISTRY.build('prithvi_eo_v2_300')
        self.neck     = MultiScaleNeck(n_side=14, d_model=1024, fpn_ch=256)
        self.decoder  = FPNDecoder(in_ch=256)
    def forward(self, x):
        return self.decoder(self.neck(self.backbone(x.unsqueeze(2))))


In [ ]:
# [6] Load the trained checkpoint (no training here, inference only)
model = PrithviSegModelV2().to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
print(f'Checkpoint loaded: {CKPT.name}')


In [ ]:
# [7] Evaluation helper -- accumulates a confusion-matrix sweep across many thresholds
# WITHOUT ever holding all pixels in memory (works patch-by-patch, running totals only).
THRESHOLDS = np.round(np.arange(0.0, 1.01, 0.02), 3)   # 51 points, fine enough for AUC via trapezoid

class PairDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths, self.mask_paths = img_paths, mask_paths
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, idx):
        raw  = np.load(self.img_paths[idx],  mmap_mode='r').astype(np.float32)
        mask = np.load(self.mask_paths[idx], mmap_mode='r')
        img  = (raw[BAND_IDX] / 10000.0 - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        img  = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        mc   = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy().astype(np.uint8)
        return torch.from_numpy(img), torch.from_numpy(mc)


def evaluate_site(site_name, patches_dir, batch_size=16, num_workers=2):
    img_dir  = patches_dir / 'images'
    mask_dir = patches_dir / 'masks_dnbr'

    if IN_COLAB:
        local = Path(f'/content/{site_name}_patches')
        local_img, local_mask = local / 'images', local / 'masks_dnbr'
        if not (local_img.exists() and any(local_img.glob('*.npy'))):
            local_img.mkdir(parents=True, exist_ok=True)
            local_mask.mkdir(parents=True, exist_ok=True)
            for src, dst in [(img_dir, local_img), (mask_dir, local_mask)]:
                for attempt in range(3):
                    result = subprocess.run(['rsync', '-a', f'{src}/', f'{dst}/'])
                    if result.returncode == 0:
                        break
                    print(f'  retry {attempt+1} (exit {result.returncode})...')
                else:
                    raise RuntimeError(f'Could not copy {src} after 3 attempts')
        img_dir, mask_dir = local_img, local_mask

    img_paths  = sorted(img_dir.glob('*.npy'))
    mask_paths = sorted(mask_dir.glob('*.npy'))
    assert len(img_paths) == len(mask_paths) and img_paths, f'{site_name}: no paired patches found'
    print(f'{site_name}: {len(img_paths)} patches')

    ds     = PairDataset(img_paths, mask_paths)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)

    n_t = len(THRESHOLDS)
    tp = np.zeros(n_t, dtype=np.int64)
    fp = np.zeros(n_t, dtype=np.int64)
    fn = np.zeros(n_t, dtype=np.int64)
    tn = np.zeros(n_t, dtype=np.int64)

    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc=f'{site_name} inference'):
            probs = model(imgs.to(DEVICE)).sigmoid().cpu().numpy()  # (B,1,H,W)
            targets = masks.numpy().astype(np.uint8)                # (B,H,W)
            p = probs.reshape(probs.shape[0], -1)
            g = targets.reshape(targets.shape[0], -1)
            for i, t in enumerate(THRESHOLDS):
                pred = (p > t)
                gt   = (g == 1)
                tp[i] += int((pred & gt).sum())
                fp[i] += int((pred & ~gt).sum())
                fn[i] += int((~pred & gt).sum())
                tn[i] += int((~pred & ~gt).sum())

    iou  = tp / (tp + fp + fn + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    tpr  = tp / (tp + fn + 1e-9)
    fpr  = fp / (fp + tn + 1e-9)

    # AUC via trapezoidal integration of the swept ROC curve (thresholds descending
    # -> FPR ascending, required for np.trapz to integrate left-to-right correctly)
    order = np.argsort(fpr)
    auc = float(np.trapezoid(tpr[order], fpr[order]))

    bi = int(iou.argmax())
    fixed_i = int(np.argmin(np.abs(THRESHOLDS - 0.450)))

    result = {
        'site': site_name,
        'n_patches': len(img_paths),
        'fixed_threshold': {
            't': float(THRESHOLDS[fixed_i]), 'iou': float(iou[fixed_i]),
            'precision': float(prec[fixed_i]), 'recall': float(rec[fixed_i]),
            'f1': float(f1[fixed_i]),
        },
        'best_threshold': {
            't': float(THRESHOLDS[bi]), 'iou': float(iou[bi]),
            'precision': float(prec[bi]), 'recall': float(rec[bi]),
            'f1': float(f1[bi]),
        },
        'auc_roc': auc,
        'sweep': [
            {'t': float(THRESHOLDS[i]), 'iou': float(iou[i]), 'precision': float(prec[i]),
             'recall': float(rec[i]), 'f1': float(f1[i])}
            for i in range(n_t)
        ],
    }
    print(f'{site_name}: fixed(t=0.450) IoU={iou[fixed_i]:.4f}  '
          f'best(t={THRESHOLDS[bi]:.2f}) IoU={iou[bi]:.4f}  AUC-ROC={auc:.4f}')
    return result

print('Evaluation helper ready.')


In [ ]:
# [8] Run for all 3 sites, save results, print comparison against the old v2.0 numbers
OLD_V20 = {
    'cordoba': {'iou': 0.115, 'precision': 0.20,  'recall': 0.21, 'auc_roc': 0.738},
    'greece':  {'iou': 0.234, 'precision': 0.481, 'recall': 0.31, 'auc_roc': 0.652},
    'canada':  {'iou': 0.191, 'precision': 0.680, 'recall': 0.21, 'auc_roc': 0.606},
}

results = {}
for site_name, patches_dir in SITES.items():
    results[site_name] = evaluate_site(site_name, patches_dir)

out_path = OUTPUTS_V22 / 'zs_reeval_v22.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved: {out_path}')

print('\n' + '=' * 78)
print(f'{"Site":<10} {"v2.0 IoU":>9} {"v2.2 IoU (fixed)":>17} {"v2.2 IoU (best)":>16} {"v2.0 AUC":>9} {"v2.2 AUC":>9}')
print('-' * 78)
for site_name in SITES:
    old = OLD_V20[site_name]
    new = results[site_name]
    print(f'{site_name:<10} {old["iou"]:>9.4f} {new["fixed_threshold"]["iou"]:>17.4f} '
          f'{new["best_threshold"]["iou"]:>16.4f} {old["auc_roc"]:>9.4f} {new["auc_roc"]:>9.4f}')
print('=' * 78)
print('If v2.2 fixed-threshold IoU > v2.0 IoU across sites, the Australia biome addition')
print('is verified to help zero-shot generalization broadly, not just assumed.')
print('DONE. This file (zs_reeval_v22.json) is what the local write-up step needs next.')


In [ ]:
import json
import numpy as np
from pathlib import Path

# Chile no tiene baseline v2.0 (fue el primer sitio evaluado recien en v2.2)
chile = {
    "iou_fixed": 0.175, "precision": 0.218, "recall": 0.472,
    "auc_roc": 0.855, "best_possible_iou": 0.178,
    "n_pixels": 116976640, "n_polygons": 146,
    "source": "16_chile_prob_export_v22.ipynb, dNBR>0.15, T19HBD",
}

# Chequeo critico: % de pixeles positivos en el ground truth de cada sitio.
# Si canada da un positive rate alto, el best_threshold=0.00 es un optimo
# trivial (predecir todo positivo), no una mejora real de discriminacion.
pos_rate = {}
for site, patches_dir in SITES.items():
    pdir = Path(patches_dir)
    mask_files = sorted(pdir.glob("masks_dnbr/*.npy")) or sorted(pdir.parent.glob("masks_dnbr/*.npy"))
    if not mask_files:
        pos_rate[site] = None
        continue
    total = pos = 0
    for fp in mask_files:
        m = np.load(fp)
        total += m.size
        pos += int((m > 0).sum())
    pos_rate[site] = pos / total

print("Ground-truth positive rate por sitio (fraccion de pixeles quemados):")
for site, r in pos_rate.items():
    flag = "  <-- si es alto, best_threshold=0.00 es sospechoso de ser trivial" if site == "canada" else ""
    print(f"  {site:<10} {r if r is not None else 'N/A -- ajustar el path del glob'}{flag}")

print("\n" + "="*100)
print(f"{'site':<10}{'v2.0 IoU':>10}{'v2.2 IoU(fix)':>15}{'v2.2 IoU(best)':>16}{'best_t':>8}{'v2.0 AUC':>10}{'v2.2 AUC':>10}{'pos_rate':>10}")
print("-"*100)
for site in SITES:
    old, new = OLD_V20[site], results[site]
    bt = new["best_threshold"].get("t", new["best_threshold"].get("threshold", "?"))
    pr = pos_rate.get(site)
    print(f"{site:<10}{old['iou']:>10.4f}{new['fixed_threshold']['iou']:>15.4f}{new['best_threshold']['iou']:>16.4f}"
          f"{bt!s:>8}{old['auc_roc']:>10.4f}{new['auc_roc']:>10.4f}{(pr if pr is not None else float('nan')):>10.3f}")
print(f"{'chile':<10}{'N/A':>10}{chile['iou_fixed']:>15.4f}{chile['best_possible_iou']:>16.4f}"
      f"{'~0.44':>8}{'N/A':>10}{chile['auc_roc']:>10.4f}{'?':>10}")
print("="*100)

export = {
    "v22_reeval": {s: results[s] for s in SITES},
    "v20_baseline": {s: OLD_V20[s] for s in SITES},
    "gt_positive_rate": pos_rate,
    "chile_v22": chile,
}
out_path = Path("/content/drive/MyDrive/GeoAI/wildfire-spread/outputs/v2.2/full_zs_comparison_v22.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(export, f, indent=2, default=str)
print(f"\nGuardado -> {out_path}")
